In [1]:
import sys
# sys.path.append('../')

In [2]:
import os
import sys
import torch
import accelerate 
import logging
import hydra
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader
from diffusers.optimization import get_cosine_schedule_with_warmup

from utils.logging import setup_logging
from torch import optim
from rich.progress import Progress, BarColumn, TextColumn, TimeElapsedColumn, TimeRemainingColumn
from rich.syntax import Syntax


/home/nero/miniforge3/envs/dawn/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with hydra.initialize(config_path="configs"):
    cfg = hydra.compose(config_name="stage1")
    OmegaConf.resolve(cfg)


In [4]:
accelerator = accelerate.Accelerator(**cfg.accelerator)

setup_logging(accelerator.is_main_process, log_dir=cfg.trainer.save_dir)

logger = logging.getLogger(__name__)
logger.info("Configuration:\n" + OmegaConf.to_yaml(cfg))

# # Init dataset
train_dataset = hydra.utils.instantiate(cfg.dataset.train)
val_dataset = hydra.utils.instantiate(cfg.dataset.val)

# Init dataloader
train_loader = DataLoader(
    train_dataset, 
    batch_size=cfg.loader.batch_size, 
    shuffle=True, 
    num_workers=cfg.loader.num_workers
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.loader.batch_size,
    shuffle=False,
    num_workers=cfg.loader.num_workers
)

model = hydra.utils.instantiate(cfg.model)


06/12/2025 17:14:01 INFO     [06/12/2025 17:14:01] {__main__} - Configuration:                                     
                             project: DAWN_stage_1                                                                 
                             debug: false                                                                          
                             accelerator:                                                                          
                               gradient_accumulation_steps: 1                                                      
                               mixed_precision: fp16                                                               
                               log_with: wandb                                                                     
                               project_dir: ./outputs                                                              
                             loader:                                                                               
                               batch_size: 32                                                                      
                               num_workers: 4                                                                      
                             trainer:                                                                              
                               accumulate: 1                                                                       
                               total_steps: 50000                                                                  
                               log_interval: 20                                                                    
                               save_interval: 1000                                                                 
                               val_interval: 1000                                                                  
                               save_dir: ./outputs                                                                 
                               resume: false                                                                       
                               lr_warmup_steps: 10                                                                 
                             optimizer:                                                                            
                               _target_: torch.optim.AdamW                                                         
                               lr: 0.0001                                                                          
                               weight_decay: 0.01                                                                  
                               betas:                                                                              
                               - 0.9                                                                               
                               - 0.999                                                                             
                             seed: 0                                                                               
                             dataset:                                                                              
                               train:                                                                              
                                 _target_: dawn.data.calvin.calvin.CalvinDataset                                   
                                 data_path: data/calvin/dataset_opt/task_ABC_D                                     
                                 split: training                                                                   
                               val:                                                                                
                                 _target_: dawn.data.cal

06/12/2025 17:14:01 INFO     [06/12/2025 17:14:01] {__main__} - Configuration:                                          
                             project: DAWN_stage_1                                                                      
                             debug: false                                                                               
                             accelerator:                                                                               
                               gradient_accumulation_steps: 1                                                           
                               mixed_precision: fp16                                                                    
                               log_with: wandb                                                                          
                               project_dir: ./outputs                                                                   
                             loader:                                                                                    
                               batch_size: 32                                                                           
                               num_workers: 4                                                                           
                             trainer:                                                                                   
                               accumulate: 1                                                                            
                               total_steps: 50000                                                                       
                               log_interval: 20                                                                         
                               save_interval: 1000                                                                      
                               val_interval: 1000                                                                       
                               save_dir: ./outputs                                                                      
                               resume: false                                                                            
                               lr_warmup_steps: 10                                                                      
                             optimizer:                                                                                 
                               _target_: torch.optim.AdamW                                                              
                               lr: 0.0001                                                                               
                               weight_decay: 0.01                                                                       
                               betas:                                                                                   
                               - 0.9                                                                                    
                               - 0.999                                                                                  
                             seed: 0                                                                                    
                             dataset:                                                                                   
                               train:                                                                                   
                                 _target_: dawn.data.calvin.calvin.CalvinDataset                                        
                                 data_path: data/calvin/dataset_opt/task_ABC_D                                          
                                 split: training                                                                        
       

/home/nero/miniforge3/envs/dawn/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


06/12/2025 17:14:42 INFO     [06/12/2025 17:14:42] {dawn.data.calvin.calvin} - Loaded 17870 episodes from          
                             data/calvin/dataset_opt/task_ABC_D/training/episodes

06/12/2025 17:14:42 INFO     [06/12/2025 17:14:42] {dawn.data.calvin.calvin} - Loaded 17870 episodes from               
                             data/calvin/dataset_opt/task_ABC_D/training/episodes

...

06/12/2025 17:14:48 INFO     [06/12/2025 17:14:48] {dawn.models.system1.langtomo} - Initializing MotionEstimation  
                             with image size 128, in_channels 7, out_channels 2, condition_dim 512, size B,        
                             flow_to_rgb False.

06/12/2025 17:14:48 INFO     [06/12/2025 17:14:48] {dawn.models.system1.langtomo} - Initializing MotionEstimation with  
                             image size 128, in_channels 7, out_channels 2, condition_dim 512, size B, flow_to_rgb      
                             False.

2025-06-12 17:14:49.874618: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-12 17:14:49.942928: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-12 17:14:52.012108: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


06/12/2025 17:14:56 INFO     [06/12/2025 17:14:56] {dawn.models.system1.langtomo} - <All keys matched successfully>

06/12/2025 17:14:56 INFO     [06/12/2025 17:14:56] {dawn.models.system1.langtomo} - <All keys matched successfully>

In [5]:

state_dict = torch.load('outputs/checkpoints/model_0026000.pth')

/tmp/ipykernel_1386872/1830816439.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('outputs/checkpoints/model_0026000.pth')


In [6]:
model.load_state_dict(state_dict, strict=True)

<All keys matched successfully>

In [5]:
model, train_loader, val_loader = accelerator.prepare(model, train_loader, val_loader)

In [6]:
model.eval()

MotionEstimation(
  (flow_model): RAFT(
    (feature_encoder): FeatureEncoder(
      (convnormrelu): Conv2dNormActivation(
        (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): ReLU(inplace=True)
      )
      (layer1): Sequential(
        (0): ResidualBlock(
          (convnormrelu1): Conv2dNormActivation(
            (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (2): ReLU(inplace=True)
          )
          (convnormrelu2): Conv2dNormActivation(
            (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (2): ReLU(inplace=True)
          )
          (downsample): Identity()
      

In [14]:
from dawn.models.system1.flow_utils import FlowNormalizer, visualize_flow_vectors_as_PIL
import numpy as np

In [ ]:
from diffusers import DDIMScheduler
import matplotlib.pyplot as plt
num_inference_steps = 100
for batch_data in val_loader:
    scheduler = DDIMScheduler(num_train_timesteps=1000)  # You can adjust timesteps based on your model's training
    scheduler.set_timesteps(num_inference_steps)

    gt_rgb_flow = model.gen_flow(batch_data["rgb_static"])
    model.flow_to_rgb = True
    gt_rgb_flow_rgb = model.gen_flow(batch_data["rgb_static"])
    model.flow_to_rgb = False

    norm_rgb = (batch_data["rgb_static"] - model.mean) / model.std

    norm_rgb = norm_rgb[:, 0]  # Use the first frame for conditioning
    text = batch_data["language"]
    text_condition = batch_data["language_embedding"].unsqueeze(1)  # Assuming text_condition is already precomputed and passed in the batch_data
        

    bs = norm_rgb.shape[0]
    start_flow = torch.randn(gt_rgb_flow.shape, device=gt_rgb_flow.device)
    latents = start_flow.clone()

    # Iterate through DDIM timesteps
    for t in scheduler.timesteps:
        # Prepare the model inputs
        with torch.no_grad():
            # Predict the noise (epsilon) using the model
            prev_flow = gt_rgb_flow
            prev_flow  = torch.ones_like(prev_flow) * 0.5
            model_input = torch.concat([latents, norm_rgb, prev_flow], dim=1)
            time_step = torch.ones(latents.shape[0], dtype=torch.int64, device=latents.device) * t
            predicted_noise = model.model(model_input, time_step, text_condition, return_dict=False)[0]

        # Update the latent based on DDIM step
        latents = scheduler.step(predicted_noise, t, latents).prev_sample

    # The final latent is the denoised output
    generated_flow = latents

    if batch_data["rgb_static"].shape[1] > 1:
        gt_rgb_flow = model.gen_flow(batch_data["rgb_static"])
    
        with torch.no_grad():
            loss = torch.mean((generated_flow - gt_rgb_flow) ** 2)

    gt_rgb_flow_rgb = ((gt_rgb_flow_rgb * model.std) + model.mean).clamp(0, 255).to(torch.uint8) 

    generated_flow_np = generated_flow.permute(0, 2, 3, 1).cpu().numpy()
    gt_rgb_flow_np = gt_rgb_flow.permute(0, 2, 3, 1).cpu().numpy()
    gt_rgb_flow_rgb_np = gt_rgb_flow_rgb.permute(0, 2, 3, 1).cpu().numpy()

    normalizer = FlowNormalizer(model.image_size, model.image_size)

    for i in range(4):
        print(f"Batch {i} - {batch_data['language'][i]}")
        plt.subplot(2, 2, 1)
        images_np = batch_data["rgb_static"][i, 0].permute(1, 2, 0).cpu().numpy()
        plt.imshow(images_np)
        plt.subplot(2, 2, 2)
        plt.imshow(gt_rgb_flow_rgb_np[i])
        plt.subplot(2, 2, 3)
        gt_flow = normalizer.unnormalize(gt_rgb_flow_np[i])
        print("GT", np.min(gt_flow), np.max(gt_flow))
        gt = visualize_flow_vectors_as_PIL(images_np, gt_flow, step=4, title="Ground Truth Optical Flow")
        plt.imshow(gt)
        # plt.imshow(batch_data["rgb_static"][i, 1].permute(1, 2, 0).cpu().numpy())
        plt.subplot(2, 2, 4)
        pd_flow = normalizer.unnormalize(generated_flow_np[i])
        print("Generated", np.min(pd_flow), np.max(pd_flow))
        generated = visualize_flow_vectors_as_PIL(images_np, pd_flow, step=4, title="Generated Optical Flow")   
        plt.imshow(generated)
        plt.suptitle(f"Loss: {loss.item():.4f}, Text: {batch_data['language'][i]}")
        plt.tight_layout()
        plt.show()
    break 
    

0.42253616 0.5695637
